# Lab 2 (TD) — Time Series Forecasting with LSTM (SYNOP Weather)

**Objective:** *Guess what the weather will be like!*  
We will build an **LSTM** model to predict the **next-hour air temperature** from recent meteorological observations.

**Dataset:** SYNOP meteorological data (CSV) — example station: *LYS*.

> This notebook is written to be **independent from FIDLE**.  
> You only need Python + common ML libraries.

---

## What you will do
1. Load and clean the CSV  
2. Build a supervised time-series dataset (sliding windows)  
3. Train an LSTM regression model  
4. Evaluate and visualize predictions  
5. (Optional) Extend to multi-step forecasting or “weather state” classification


## 0) Setup

### Install (if needed)
Run this **once** in your environment (terminal or notebook cell):

```bash
pip install -U pandas numpy matplotlib scikit-learn keras
```

- If you prefer **Keras 3 with the Torch backend**, install torch too:
```bash
pip install -U torch
```


In [1]:
pip install keras

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\fidle-tp\fidle-env\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
pip install tensorflow

ERROR: Could not install packages due to an OSError: [Errno 28] No space left on device

You should consider upgrading via the 'c:\fidle-tp\fidle-env\Scripts\python.exe -m pip install --upgrade pip' command.



     ------------------------------------ 331.7/331.7 MB 959.4 kB/s eta 0:00:00
  Using cached tensorboard-2.20.0-py3-none-any.whl (5.5 MB)
     -------------------------------------- 437.0/437.0 KB 1.1 MB/s eta 0:00:00
  Using cached keras-3.10.0-py3-none-any.whl (1.4 MB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl (2.4 kB)
  Attempting uninstall: tensorboard-data-server
    Found existing installation: tensorboard-data-server 0.6.1
    Uninstalling tensorboard-data-server-0.6.1:
      Successfully uninstalled tensorboard-data-server-0.6.1
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.6
    Not uninstalling protobuf at c:\users\kered\appdata\local\packages\pythonsoftwarefoundation.python.3.9_qbz5n2kfra8p0\localcache\local-packages\python39\site-packages, outside environment c:\fidle-tp\fidle-env
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: tensorboard
    Found existing installation: tens

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Keras (works with TF backend or Keras 3 backends)
# If you want torch backend, uncomment the next 2 lines BEFORE importing keras:
# os.environ["KERAS_BACKEND"] = "torch"
import keras
from keras import layers

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("Keras version:", keras.__version__)


## 1) Load the SYNOP dataset

### Set the CSV path
Edit `CSV_PATH` to point to your file, for example:

`C:\Users\Kered\ML_DVRC_2025\Lecture9\Lab2\donnees-synop-essentielles-omm-LYS.csv`

> Note: This CSV uses a **semicolon `;`** separator.


In [ ]:
# ---- EDIT THIS PATH ----
CSV_PATH = r"C:\Users\Kered\ML_DVRC_2025\Lecture9\Lab2\donnees-synop-essentielles-omm-LYS.csv"

df = pd.read_csv(CSV_PATH, sep=";", low_memory=False)
print("Shape:", df.shape)
df.head()


## 2) Basic cleaning + time index

We will:
- Parse the `Date` column as a datetime
- Sort by time
- Keep a subset of useful numeric features
- Create the prediction target: **temperature at t+1 hour**

> In the SYNOP export, `Température` is often expressed in **Kelvin**.  
> We convert it to **°C** using: `temp_C = Température - 273.15`.


In [ ]:
# Parse datetime (timezone-aware strings are supported)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# Convert temperature to Celsius (most SYNOP exports are in Kelvin)
df["temp_C"] = df["Température"] - 273.15

# Candidate numeric features (you can add/remove)
feature_cols = [
    "temp_C",
    "Humidité",
    "Pression au niveau mer",
    "Point de rosée",
    "Vitesse du vent moyen 10 mn",
    "Direction du vent moyen 10 mn",
    "Nebulosité totale",
    "Précipitations dans la dernière heure",
    "Précipitations dans les 3 dernières heures",
    "Précipitations dans les 6 dernières heures",
    "Précipitations dans les 12 dernières heures",
    "Précipitations dans les 24 dernières heures",
]

# Keep only columns that exist in your file
feature_cols = [c for c in feature_cols if c in df.columns]
print("Using features:", feature_cols)

data = df[["Date"] + feature_cols].copy()

# Replace missing values (simple baseline)
# Option 1: forward-fill then back-fill
data[feature_cols] = data[feature_cols].ffill().bfill()

# Create target = next-hour temperature
data["target_temp_next_C"] = data["temp_C"].shift(-1)

# Drop last row (no target available)
data = data.dropna(subset=["target_temp_next_C"]).reset_index(drop=True)

data.head()


## 3) Quick visualization

This helps you verify:
- The time axis looks correct
- Temperature is in a reasonable range


In [ ]:
plt.figure()
plt.plot(data["Date"], data["temp_C"])
plt.title("Temperature (°C) over time")
plt.xlabel("Date")
plt.ylabel("temp_C")
plt.show()

data[feature_cols + ["target_temp_next_C"]].describe().T


## 4) Create sliding windows (supervised dataset)

We will predict **t+1 hour** using the last **LOOKBACK** hours.

- `LOOKBACK = 24` means: use the last 24 hours to predict the next hour.


In [ ]:
LOOKBACK = 24  # hours (rows)
HORIZON = 1    # predict 1 step ahead

X_all = data[feature_cols].to_numpy(dtype=np.float32)
y_all = data["target_temp_next_C"].to_numpy(dtype=np.float32)

# Chronological split (no shuffling!)
n = len(data)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

X_train_raw, y_train_raw = X_all[:train_end], y_all[:train_end]
X_val_raw,   y_val_raw   = X_all[train_end:val_end], y_all[train_end:val_end]
X_test_raw,  y_test_raw  = X_all[val_end:], y_all[val_end:]

# Scale features using train statistics only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled   = scaler.transform(X_val_raw)
X_test_scaled  = scaler.transform(X_test_raw)

def make_windows(X, y, lookback=24):
    """Convert a multivariate time series into (windows, target)."""
    Xw, yw = [], []
    for i in range(lookback, len(X)):
        Xw.append(X[i-lookback:i, :])
        yw.append(y[i])
    return np.asarray(Xw, dtype=np.float32), np.asarray(yw, dtype=np.float32)

X_train, y_train = make_windows(X_train_scaled, y_train_raw, LOOKBACK)
X_val,   y_val   = make_windows(X_val_scaled,   y_val_raw,   LOOKBACK)
X_test,  y_test  = make_windows(X_test_scaled,  y_test_raw,  LOOKBACK)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)


## 5) Build an LSTM model (regression)

We will start simple:
- LSTM → Dense → 1 output (next-hour temperature)

Loss: MSE  
Metric: MAE


In [ ]:
n_features = X_train.shape[-1]

model = keras.Sequential([
    layers.Input(shape=(LOOKBACK, n_features)),
    layers.LSTM(64),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)  # regression output
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=["mae"]
)

model.summary()


## 6) Train

We use early stopping to avoid overfitting.


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

plt.figure()
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Training curves (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


## 7) Evaluate + visualize predictions

We plot predictions vs ground truth on the test split.


In [ ]:
test_metrics = model.evaluate(X_test, y_test, verbose=0)
print("Test metrics:", dict(zip(model.metrics_names, test_metrics)))

y_pred = model.predict(X_test).reshape(-1)

plt.figure()
plt.plot(y_test[:300], label="true")
plt.plot(y_pred[:300], label="pred")
plt.title("Next-hour temperature prediction (first 300 test samples)")
plt.xlabel("Time step")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.show()

# Simple error statistics
errors = y_pred - y_test
print("MAE (°C):", np.mean(np.abs(errors)).round(3))
print("RMSE (°C):", np.sqrt(np.mean(errors**2)).round(3))


## 8) Save the model (optional)

You can save the trained model and the scaler to reuse it later.


In [ ]:
import joblib
from pathlib import Path

OUT_DIR = Path("outputs_lab2")
OUT_DIR.mkdir(exist_ok=True)

model_path = OUT_DIR / "lstm_synop_temp.keras"
scaler_path = OUT_DIR / "scaler.joblib"

model.save(model_path)
joblib.dump(scaler, scaler_path)

print("Saved:", model_path)
print("Saved:", scaler_path)


## 9) Extensions (ideas)

1. **Multi-step forecasting**: predict t+1 … t+24 (next day)  
2. **Weather state classification**: use `Temps présent` as labels (classification)  
3. **Add seasonality**: hour-of-day, day-of-year (sin/cos encoding)  
4. **Compare models**: GRU, 1D-CNN, Transformer encoder  
5. **Better missing data**: interpolation, stationarity checks, outlier filtering
